# 03 · Delineate Anything v2 — поля как отдельные объекты (YOLO11) по одному RGB-снимку

**Что это.** Модель группы Лавренюка/Куссуль/Шелестова (ECCV 2026), обученная на 73 млн полей из 61 страны,
в т.ч. по снимкам Sentinel-2. В отличие от FTW она сразу выдаёт **отдельные поля-полигоны** (сегментация
экземпляров), работает с любым разрешением 0,25–10 м и уже умеет склеивать поля через швы тайлов и
убирать «гигантов». Авторы прогнали её на всю Украину (ближайший к нашим степям ландшафт).
Код и веса: <https://github.com/Lavreniuk/Delineate-Anything>, лицензия **AGPL-3.0** (для внутренних и
исследовательских задач ограничений нет; при распространении изменённого кода или работе как веб-сервис
нужно открыть исходники).

**Что делает ноутбук.**
1. Собирает вход: либо **медианный RGB Sentinel-2** за окно середины сезона в GEE (по умолчанию; можно несколько
   окон — модель запустится на каждом, результаты сравните), либо **ваш композит** (3 канала max/min/diff NDVI —
   модель на таком не обучалась, но проверить стоит: это минуты).
2. Запускает `delineate.py` из официального репозитория на GPU.
3. Склеивает блоки, фильтрует по доле пашни WorldCover, сохраняет GeoPackage и растр границ для гибрида с v7.
4. (по желанию) Сравнивает с вашей разметкой.

**Среда:** Среда выполнения → Сменить среду → **T4 GPU**. Скорость: вся Украина ≈5 ч на одной видеокарте 16 ГБ,
т.е. область размера Татарстана — десятки минут.

**Чего ждать.** Независимая проверка (США, Sentinel-2 10 м) дала объектную F1 ≈0,34; сопровождающий пишет, что
вне похожих на Европу территорий без дообучения результат слабее. Для наших степей и лесостепи — проверяем.

## Шаг 0. Установка (один раз за сеанс, ~2 мин)
Клонируется официальный репозиторий и ставятся ultralytics/huggingface_hub. Библиотеки Colab (rasterio, shapely,
GDAL) не трогаются, поэтому перезапуск среды обычно не нужен.

In [ ]:
import os, subprocess, sys, importlib.util
DA_DIR = '/content/Delineate-Anything' if not os.environ.get('LOCAL_TEST') else os.environ.get('DA_DIR', '/tmp/Delineate-Anything')
if not os.path.exists(DA_DIR):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/Lavreniuk/Delineate-Anything.git', DA_DIR], check=True)
need = [p for p in ('ultralytics', 'huggingface_hub', 'numba', 'yaml') if importlib.util.find_spec(p) is None]
if need:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.3.148', 'huggingface_hub', 'numba', 'pyyaml'], check=True)
try:
    from osgeo import gdal
    print('GDAL', gdal.__version__)
except ImportError:
    subprocess.run('apt-get -qq install -y python3-gdal gdal-bin > /dev/null', shell=True)
    print('GDAL установлен из apt; если ниже будет ошибка импорта osgeo — перезапустите сеанс')
import torch
print('репозиторий:', DA_DIR, '| GPU:', torch.cuda.is_available())

### Подключение Google Диска
Выполните ячейку и разрешите доступ к Диску (всплывёт окно Google). Все результаты пишутся в папку
`Colab_Notebooks/Agro_terra/research/…` — её можно поменять в параметрах ниже.

In [ ]:
import os, sys, json, time, glob, shutil, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
LOCAL_TEST = bool(os.environ.get('LOCAL_TEST'))          # True только при проверке вне Colab
if not LOCAL_TEST:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT = Path(os.environ['LOCAL_DRIVE'])
print('Диск:', DRIVE_ROOT, '| файлов в корне:', len(list(DRIVE_ROOT.iterdir())))

## Шаг 1. Параметры — заполните под себя
**Источник снимка** `IMAGE_SOURCE`:
- `'gee'` — медиана Sentinel-2 RGB (B4, B3, B2) за окна `WINDOWS` (рекомендуется: модель обучалась на RGB);
  для каждого окна будет отдельный результат. Окно середины сезона (июнь–июль) обычно лучше всего разделяет
  соседние поля разных культур; позднее окно (август) — хорошо видны убранные/неубранные поля.
- `'composite'` — ваш 3-канальный композит из `COMPOSITE_PATH` (будет перепроецирован в UTM, как требует модель).

In [ ]:
EE_PROJECT = 'my-project-20260614-499410'
IMAGE_SOURCE = 'gee'                 # 'gee' или 'composite'
COMPOSITE_PATH = 'Colab_Notebooks/Agro_terra/TAT/TA_composite_2026_0504_0910_8b.tif'   # охват участка и сетка для растра границ
AOI_PATH = None
BBOX = None                          # (lon_min, lat_min, lon_max, lat_max) — для быстрого теста на маленьком участке
WINDOWS = {'jun_jul': ('2026-06-01', '2026-07-31')}     # можно добавить, напр. 'aug': ('2026-08-01', '2026-08-31')
BLOCK_DEG = 0.5

MODEL = 'large_v2'                   # 'large_v2' — Delineate Anything v2; 'large' / 'small' — версия 1
MIN_CONF = 0.15                      # минимальная уверенность объекта (ниже — больше полей и больше мусора)
GIANT_FILTER = True                  # фильтр «гигантов»; на очень крупных полях может удалить и настоящие — сравните с False
BATCH = 8                            # тайлов за раз на GPU (T4 16 ГБ: 8; при нехватке памяти — 4)

MIN_CROP_FRAC = 0.5
MIN_HA = 1.0
GT_PATH = None
GT_TYPE_COL = 'type'
OUT_DIR = 'Colab_Notebooks/Agro_terra/research/delineate_anything/out'
GEE_FOLDER = 'GEE_delineate_anything'
NAME = 'TAT_2026'

## Шаг 2. Участок и блоки

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, shapely, rasterio, yaml
from rasterio.warp import transform_bounds, calculate_default_transform, reproject, Resampling
OUT = DRIVE_ROOT / OUT_DIR; OUT.mkdir(parents=True, exist_ok=True)
WORK = Path('/content/da_work') if not LOCAL_TEST else Path(os.environ.get('DA_WORK', '/tmp/da_work'))
IMG_ROOT = OUT / f'{NAME}_input'; IMG_ROOT.mkdir(parents=True, exist_ok=True)
if BBOX:
    AOI_BBOX = tuple(BBOX)
elif COMPOSITE_PATH:
    with rasterio.open(DRIVE_ROOT / COMPOSITE_PATH) as s:
        AOI_BBOX = transform_bounds(s.crs, 'EPSG:4326', *s.bounds)
else:
    AOI_BBOX = tuple(gpd.read_file(DRIVE_ROOT / AOI_PATH).to_crs(4326).total_bounds)
print('рамка', np.round(AOI_BBOX, 3))

## Шаг 3. Входные снимки
**Вариант `gee`.** Для каждого окна — медиана безоблачных пикселей (Cloud Score+), каналы B4,B3,B2 переводятся
в 8 бит как «истинные цвета» Sentinel-2 (отражение 0–0,3 → 0–255), проекция UTM 10 м, блоки с перекрытием.
Ячейка ждёт окончания экспорта; её можно прервать и перезапустить — готовые блоки повторно не экспортируются.

**Вариант `composite`.** Композит перепроецируется в UTM 10 м (модель требует метровую проекцию).

In [ ]:
ENTRIES = {}          # имя запуска → папка с тифами

Следующая ячейка (GEE) нужна только для варианта `IMAGE_SOURCE = 'gee'`; для `'composite'` её можно пропустить.

### Google Earth Engine: подключение и функции сборки многодатных композитов (общие для ноутбуков)
При первом запуске всплывёт окно авторизации Google — разрешите доступ для вашего проекта GEE.

Что умеют функции ниже:
- `s2_collection(aoi, start, end)` — Sentinel-2 L2A **HARMONIZED** (значения уже без смещения +1000, как ждут модели FTW),
  облака/тени маскируются по **Cloud Score+** (`cs_cdf ≥ CS_THR`);
- `window_median(aoi, start, end, bands)` — медиана за окно дат; пропуски (облака) заполняются медианой расширенного окна;
- `monthly_ndvi(aoi, year, months)` — помесячные медианы NDVI (многодатный стек для классических методов);
- `export_blocks(img, aoi, prefix, …)` — экспорт на Google Диск блоками (большой участок режется на квадраты с перекрытием);
- `wait_tasks(tasks)` — ждать окончания экспортов и показывать прогресс.

In [ ]:
import ee, math
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print('GEE OK, проект', EE_PROJECT)

S2 = 'COPERNICUS/S2_SR_HARMONIZED'
CSP = 'GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED'
CS_THR = 0.60          # порог Cloud Score+: 0.5 — мягче (больше данных), 0.65 — строже (меньше облаков)


def s2_collection(aoi, start, end, max_cloud=80):
    col = (ee.ImageCollection(S2).filterBounds(aoi).filterDate(start, end)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud))
           .linkCollection(ee.ImageCollection(CSP), ['cs_cdf']))
    return col.map(lambda im: im.updateMask(im.select('cs_cdf').gte(CS_THR)))


def _shift(date, days):
    return ee.Date(date).advance(days, 'day')


def window_median(aoi, start, end, bands, pad_days=30):
    """Медиана за окно [start, end); дыры от облаков заполняются медианой окна, расширенного на ±pad_days."""
    core = s2_collection(aoi, start, end).select(bands).median()
    wide = s2_collection(aoi, _shift(start, -pad_days), _shift(end, pad_days)).select(bands).median()
    return core.unmask(wide)


def monthly_ndvi(aoi, year, months):
    """Многоканальный образ: NDVI-медиана каждого месяца (каналы ndvi_MM), 0–1 → ×10000 int16."""
    out = []
    for m in months:
        start = ee.Date.fromYMD(year, m, 1); end = start.advance(1, 'month')
        nd = s2_collection(aoi, start, end).map(lambda im: im.normalizedDifference(['B8', 'B4']).rename('ndvi')).median()
        out.append(nd.rename(f'ndvi_{m:02d}'))
    return ee.Image.cat(out).multiply(10000).round().toInt16()


def utm_epsg(lon, lat):
    return (32600 if lat >= 0 else 32700) + int((lon + 180) // 6) + 1


def block_grid(bbox, block_deg=0.5, overlap_deg=0.02):
    """Квадраты (по долготе/широте) с перекрытием: [(id, core_bbox, export_bbox)]."""
    x0, y0, x1, y1 = bbox; out = []
    nx, ny = max(1, math.ceil((x1 - x0) / block_deg)), max(1, math.ceil((y1 - y0) / block_deg))
    for i in range(nx):
        for j in range(ny):
            cx0, cy0 = x0 + i * block_deg, y0 + j * block_deg
            core = (cx0, cy0, min(cx0 + block_deg, x1), min(cy0 + block_deg, y1))
            exp = (core[0] - overlap_deg, core[1] - overlap_deg, core[2] + overlap_deg, core[3] + overlap_deg)
            out.append((f'b{i:02d}_{j:02d}', core, exp))
    return out


def export_blocks(img, bbox, prefix, gee_folder, scale=10, block_deg=0.5, dtype='uint16', crs=None):
    """Экспорт изображения на Диск блоками; возвращает список (id, core_bbox, имя_файла, задача).
    Файлы появятся в «Мой диск/<gee_folder>/<prefix>_<id>.tif». crs по умолчанию — UTM центра участка."""
    crs = crs or f'EPSG:{utm_epsg((bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2)}'
    img = {'uint16': img.toUint16(), 'uint8': img.toUint8(), 'int16': img.toInt16(), 'float': img.toFloat()}[dtype]
    jobs = []
    for bid, core, exp in block_grid(bbox, block_deg):
        name = f'{prefix}_{bid}'
        task = ee.batch.Export.image.toDrive(image=img.clip(ee.Geometry.Rectangle(list(exp))), description=name[:100],
                                             folder=gee_folder, fileNamePrefix=name, region=ee.Geometry.Rectangle(list(exp)),
                                             scale=scale, crs=crs, maxPixels=1e13, fileFormat='GeoTIFF',
                                             fileDimensions=16384)     # один файл на блок (без нарезки на части)
        task.start(); jobs.append((bid, core, name, task))
    print(f'Запущено экспортов: {len(jobs)} (CRS {crs}); следить можно и на https://code.earthengine.google.com/tasks')
    return jobs


def wait_tasks(jobs, poll=30):
    """Ждёт завершения всех задач экспорта. Можно прервать ячейку и вернуться позже — задачи идут на серверах GEE."""
    t0 = time.time()
    while True:
        st = [j[3].status()['state'] for j in jobs]
        done = sum(s == 'COMPLETED' for s in st); bad = [j[2] for j, s in zip(jobs, st) if s in ('FAILED', 'CANCELLED')]
        print(f'\r{time.time() - t0:6.0f} с: готово {done}/{len(jobs)}' + (f', ошибки: {bad}' if bad else ''), end='')
        if done + len(bad) == len(jobs):
            print(); break
        time.sleep(poll)
    for j, s in zip(jobs, st):
        if s == 'FAILED':
            print(j[2], j[3].status().get('error_message'))


def collect_exports(jobs, gee_folder, dst_dir, timeout=900):
    """Переносит экспортированные файлы из «Мой диск/<gee_folder>» в папку проекта (Диск синхронизируется с задержкой)."""
    src_dir = DRIVE_ROOT / gee_folder; dst_dir = Path(dst_dir); dst_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time(); left = {j[2] for j in jobs}
    while left and time.time() - t0 < timeout:
        for name in list(left):
            for f in src_dir.glob(f'{name}*.tif'):                 # большой блок GEE может разрезать на части …-0000000000-0000000000.tif
                shutil.move(str(f), dst_dir / f.name)
            if list(dst_dir.glob(f'{name}*.tif')):
                left.discard(name)
        if left:
            time.sleep(15)
    print('Файлы в', dst_dir, ':', len(list(dst_dir.glob('*.tif'))), '| не найдено:', sorted(left))

In [ ]:
if IMAGE_SOURCE == 'gee':
    aoi_ee = ee.Geometry.Rectangle(list(AOI_BBOX))
    crs = f'EPSG:{utm_epsg((AOI_BBOX[0] + AOI_BBOX[2]) / 2, (AOI_BBOX[1] + AOI_BBOX[3]) / 2)}'
    jobs = []
    for wname, (a, b) in WINDOWS.items():
        rgb = window_median(aoi_ee, a, b, ['B4', 'B3', 'B2']).divide(3000).multiply(255).clamp(1, 255).toUint8()
        for bid, core, exp in block_grid(AOI_BBOX, BLOCK_DEG):
            entry = f'{NAME}_{wname}_{bid}'; d = IMG_ROOT / entry; d.mkdir(exist_ok=True); ENTRIES[entry] = (d, core, wname)
            if list(d.glob('*.tif')):
                continue
            reg = ee.Geometry.Rectangle(list(exp))
            t = ee.batch.Export.image.toDrive(image=rgb.clip(reg), description=entry, folder=GEE_FOLDER, fileNamePrefix=entry,
                                              region=reg, scale=10, crs=crs, maxPixels=1e13, fileDimensions=16384)
            t.start(); jobs.append((bid, core, entry, t))
    if jobs:
        wait_tasks(jobs)
        for bid, core, entry, t in jobs:
            collect_exports([(bid, core, entry, t)], GEE_FOLDER, IMG_ROOT / entry)
else:
    entry = f'{NAME}_composite'; d = IMG_ROOT / entry; d.mkdir(exist_ok=True); ENTRIES[entry] = (d, None, 'composite')
    dst_tif = d / f'{entry}.tif'
    if not dst_tif.exists():
        with rasterio.open(DRIVE_ROOT / COMPOSITE_PATH) as s:
            cx, cy = (AOI_BBOX[0] + AOI_BBOX[2]) / 2, (AOI_BBOX[1] + AOI_BBOX[3]) / 2
            utm = f'EPSG:{(32600 if cy >= 0 else 32700) + int((cx + 180) // 6) + 1}'
            t, w, h = calculate_default_transform(s.crs, utm, s.width, s.height, *s.bounds, resolution=10)
            prof = dict(driver='GTiff', width=w, height=h, count=3, dtype='uint8', crs=utm, transform=t, nodata=0,
                        compress='deflate', tiled=True, BIGTIFF='IF_SAFER')
            with rasterio.open(dst_tif, 'w', **prof) as dd:
                for i in (1, 2, 3):
                    reproject(rasterio.band(s, i), rasterio.band(dd, i), resampling=Resampling.bilinear)
    print('композит в UTM:', dst_tif)
print('запусков:', len(ENTRIES))

## Шаг 4. Запуск Delineate Anything
Конфигурация берётся из `conf_sample.yaml` репозитория, меняются только: модель, каналы (1,2,3 = R,G,B),
порог уверенности, размер пачки для GPU и фильтр гигантов. Готовые результаты пропускаются.

In [ ]:
base = yaml.safe_load(open(f'{DA_DIR}/conf_sample.yaml'))
base['model'] = [MODEL]
base['data_loader'].update(bands=[1, 2, 3], nodata_value=[0, 0, 0], min=None, max=None)
base['passes'][0]['batch_size'] = BATCH
base['passes'][0]['model_args'][0].update(name=MODEL, minimal_confidence=MIN_CONF, use_half=bool(torch.cuda.is_available()))
base['giant_filter']['enabled'] = GIANT_FILTER
(WORK / 'temp').mkdir(parents=True, exist_ok=True)
conf_path = WORK / 'da_config.yaml'; yaml.safe_dump(base, open(conf_path, 'w'), allow_unicode=True)
todo = [e for e in ENTRIES if not (OUT / f'{NAME}_raw' / f'{e}.gpkg').exists() and list(ENTRIES[e][0].glob('*.tif'))]
if todo:
    batch = dict(base_config=str(conf_path), data_root=str(IMG_ROOT), output_root=str(OUT / f'{NAME}_raw'),
                 temp_root=str(WORK / 'temp'), keep_temp=False, mask_root=str(WORK / 'no_masks'), include=todo,
                 exclude=None, override=None)
    (OUT / f'{NAME}_raw').mkdir(exist_ok=True)
    bpath = WORK / 'batch.yaml'; yaml.safe_dump(batch, open(bpath, 'w'), allow_unicode=True)
    r = subprocess.run([sys.executable, 'delineate.py', '-b', str(bpath)], cwd=DA_DIR, capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-3000:])
print('готово:', sorted(p.name for p in (OUT / f'{NAME}_raw').glob('*.gpkg')))

### Постобработка (общая): доля пашни по ESA WorldCover и перенос растров на сетку композита
- `cropland_frac(gdf)` — доля пикселей WorldCover-«пашни» (класс 40) внутри каждого полигона; читается прямо из
  открытого хранилища ESA (AWS), GEE не нужен;
- `filter_fields(gdf, MIN_CROP_FRAC, MIN_HA)` — как в конвейере: оставляет полигоны, где пашни ≥ 50 % и площадь ≥ MIN_HA;
- `to_grid(src_tif, band, composite)` — переносит канал растра (например, вероятность границы) на сетку вашего композита.

In [ ]:
import math
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.windows import from_bounds
from rasterio.features import rasterize


def _wc_tiles(bbox):
    x0, y0, x1, y1 = bbox
    for lat in range(int(math.floor(y0 / 3) * 3), int(math.floor(y1 / 3) * 3) + 1, 3):
        for lon in range(int(math.floor(x0 / 3) * 3), int(math.floor(x1 / 3) * 3) + 1, 3):
            ns, ew = ('N' if lat >= 0 else 'S'), ('E' if lon >= 0 else 'W')
            yield (f'/vsicurl/https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/'
                   f'ESA_WorldCover_10m_2021_v200_{ns}{abs(lat):02d}{ew}{abs(lon):03d}_Map.tif')


def cropland_frac(gdf, agri=(40,)):
    """Доля пашни WorldCover 2021 в каждом полигоне (0–1). gdf — любая СК."""
    g = gdf.to_crs(4326); frac = np.zeros(len(g)); tot = np.zeros(len(g))
    if not len(g):
        return frac
    bbox = tuple(g.total_bounds)
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.tif'):
        for url in _wc_tiles(bbox):
            try:
                s = rasterio.open(url)
            except Exception:
                continue
            with s:
                b = (max(bbox[0], s.bounds.left), max(bbox[1], s.bounds.bottom), min(bbox[2], s.bounds.right), min(bbox[3], s.bounds.top))
                if b[0] >= b[2] or b[1] >= b[3]:
                    continue
                win = from_bounds(*b, s.transform).round_offsets().round_lengths()
                a = s.read(1, window=win); t = s.window_transform(win)
            ids = rasterize([(geom, i + 1) for i, geom in enumerate(g.geometry.values)], out_shape=a.shape, transform=t,
                            fill=0, dtype='int32')
            n = len(g) + 1
            tot += np.bincount(ids.ravel(), minlength=n)[1:]
            frac += np.bincount(ids.ravel(), weights=np.isin(a, agri).ravel().astype(float), minlength=n)[1:]
    return np.where(tot > 0, frac / np.maximum(tot, 1), 0.0)


def filter_fields(gdf, min_crop_frac=0.5, min_ha=1.0):
    g = gdf.copy()
    g['area_ha'] = g.to_crs(6933).area / 1e4
    g['crop_frac'] = cropland_frac(g)
    keep = (g.crop_frac >= min_crop_frac) & (g.area_ha >= min_ha)
    print(f'фильтр: осталось {keep.sum()} из {len(g)} (пашня ≥ {min_crop_frac:.0%}, площадь ≥ {min_ha} га)')
    return g[keep].reset_index(drop=True)


def to_grid(src_tif, band, composite, resampling=Resampling.max):
    """Канал растра → массив на сетке композита (float32, NaN — нет данных)."""
    with rasterio.open(composite) as c:
        out = np.full((c.height, c.width), np.nan, np.float32); T, crs = c.transform, c.crs
    with rasterio.open(src_tif) as s:
        reproject(rasterio.band(s, band), out, dst_transform=T, dst_crs=crs, dst_nodata=np.nan, resampling=resampling)
    return out

## Шаг 5. Склейка блоков, фильтр по пашне, сохранение
Для каждого окна дат получается свой слой в `<NAME>_DA_fields.gpkg`. Полигон относится к блоку, в «ядро»
которого попадает его центр (перекрытие блоков убирает разрезы по швам).

In [ ]:
RESULTS = {}
gpkg = OUT / f'{NAME}_DA_fields.gpkg'
for wname in sorted({v[2] for v in ENTRIES.values()}):
    parts = []
    for e, (d, core, wn) in ENTRIES.items():
        if wn != wname:
            continue
        f = OUT / f'{NAME}_raw' / f'{e}.simp.gpkg'
        f = f if f.exists() else OUT / f'{NAME}_raw' / f'{e}.gpkg'
        if not f.exists():
            continue
        g = gpd.read_file(f).to_crs(4326)
        if core is not None and len(g):
            g = g[g.representative_point().within(shapely.box(*core)).values]
        parts.append(g)
    if not parts:
        continue
    g = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=4326)
    if AOI_PATH:
        g = g[g.intersects(gpd.read_file(DRIVE_ROOT / AOI_PATH).to_crs(4326).union_all())]
    g = filter_fields(g, MIN_CROP_FRAC, MIN_HA); RESULTS[wname] = g
    g.to_file(gpkg, layer=wname, driver='GPKG')
    print(wname, ':', len(g), 'полей, медиана', round(g.area_ha.median(), 1), 'га')
print('Сохранено:', gpkg)

## Шаг 6. Растр границ на сетку композита (для гибрида с конвейером v7)
Контуры полигонов растеризуются линиями (255) на сетку `COMPOSITE_PATH` → `<NAME>_DA_<окно>_boundary.tif`.
В конвейере v7 укажите его в `EXT_BOUNDARY['TIF']`.

In [ ]:
if COMPOSITE_PATH:
    with rasterio.open(DRIVE_ROOT / COMPOSITE_PATH) as c:
        prof = dict(driver='GTiff', width=c.width, height=c.height, count=1, dtype='uint8', crs=c.crs, transform=c.transform,
                    compress='deflate', tiled=True, BIGTIFF='IF_SAFER', nodata=0)
        for wname, g in RESULTS.items():
            lines = g.to_crs(c.crs).boundary
            arr = rasterize([(l, 255) for l in lines.values if l is not None and not l.is_empty],
                            out_shape=(c.height, c.width), transform=c.transform, fill=1, dtype='uint8')
            p = OUT / f'{NAME}_DA_{wname}_boundary.tif'
            with rasterio.open(p, 'w', **prof) as d:
                d.write(arr, 1)
            print('Сохранено:', p)

### Функции оценки и картинок (общие для всех ноутбуков)
Ничего менять не нужно — просто выполните ячейку. Метрики те же, что в тестах конвейера
(`tests/eval_gt.py` в репозитории), поэтому числа можно сравнивать с v6:
- **IoU** — для каждого эталонного поля (≥ 5 га) берётся лучший по перекрытию полигон, среднее по полям;
- **IoU≥0.7** — доля эталонных полей, найденных «хорошо»;
- **frag** — сколько полигонов в среднем покрывают одно эталонное поле (1 — идеально, >1 — дробление);
- **merged** — сколько полигонов «склеили» 2 и больше эталонных полей;
- **bF** — точность границ (F-мера, допуск 2 пикселя), **P/R** — точность/полнота линий;
- **охват** — доля площади эталона, покрытая полигонами; по классам (если в эталоне есть колонка `type`).

In [ ]:
import numpy as np, geopandas as gpd, shapely, rasterio, rasterio.windows
from rasterio.features import rasterize
from rasterio.transform import from_origin
from scipy import ndimage as ndi


def load_gt(path, bbox=None):
    """Эталонная разметка: только полигоны в рамке bbox (EPSG:4326: lon_min, lat_min, lon_max, lat_max), без пустых
    и повторных. Так можно указывать разметку на всю область (например DNR_type_2026.geojson), а сравнивать на участке."""
    box = gpd.GeoSeries([shapely.box(*bbox)], crs=4326) if bbox is not None else None
    g = gpd.read_file(path, bbox=box)
    g = g[g.geometry.notna()]
    if bbox is not None:
        g = g[g.to_crs(4326).representative_point().within(box.iloc[0]).values]
    g = g[~g.geometry.to_wkb().duplicated()].reset_index(drop=True)
    print('эталон:', len(g), 'полигонов', '(в рамке участка)' if bbox is not None else '')
    return g


def grid_for(gdf, res=10.0):
    """Сетка 10 м в UTM по охвату слоя (если нет вашего композита)."""
    utm = gdf.estimate_utm_crs()
    x0, y0, x1, y1 = gdf.to_crs(utm).total_bounds
    w, h = int(np.ceil((x1 - x0) / res)) + 2, int(np.ceil((y1 - y0) / res)) + 2
    return from_origin(x0 - res, y1 + res, res, res), (h, w), utm


def evaluate(pred, gt, transform=None, shape=None, crs=None, min_ha=5, type_col='type', label=''):
    """pred, gt — GeoDataFrame полигонов (любая СК). transform/shape/crs — сетка для растровых метрик
    (по умолчанию 10 м UTM по эталону). Печатает строку метрик и возвращает dict."""
    if transform is None:
        transform, shape, crs = grid_for(gt)
    H, W = shape
    gt = gt[gt.geometry.notna()].copy(); pred = pred[pred.geometry.notna()].copy()
    gt = gt[~gt.geometry.to_wkb().duplicated()]                # в выгрузках разметки бывают повторы одного полигона
    box = shapely.box(*rasterio.transform.array_bounds(H, W, transform))
    gt = gt[gt.to_crs(crs).representative_point().within(box).values]   # эталон — только в пределах сетки (участка)
    win = rasterio.windows.from_bounds(*gt.to_crs(crs).total_bounds, transform)          # сетка — только охват эталона
    win = win.round_offsets().round_lengths()
    c0, r0 = max(int(win.col_off) - 20, 0), max(int(win.row_off) - 20, 0)
    W, H = min(int(win.col_off + win.width) + 20, W) - c0, min(int(win.row_off + win.height) + 20, H) - r0
    transform = rasterio.windows.transform(rasterio.windows.Window(c0, r0, W, H), transform)
    if H * W > 4e8:
        print(f'ВНИМАНИЕ: эталон охватывает {H}×{W} пикселей — оценка займёт много памяти; задайте участок поменьше (BBOX)')
    gt['geometry'] = shapely.make_valid(gt.geometry.values); pred['geometry'] = shapely.make_valid(pred.geometry.values)
    ea = 6933                                                  # равновеликая СК для площадей
    g6 = gt.to_crs(ea); g6 = g6[g6.area / 1e4 >= min_ha].reset_index(drop=True)
    p6 = pred.to_crs(ea); p6['geometry'] = shapely.make_valid(p6.geometry.values)
    tree = shapely.STRtree(p6.geometry.values)
    ious, frag, merged = [], [], np.zeros(len(p6), int)
    for g in g6.geometry.values:
        best, nfr = 0.0, 0
        for j in tree.query(g, predicate='intersects'):
            pj = p6.geometry.values[j]; inter = g.intersection(pj).area
            if inter >= .15 * g.area: merged[j] += 1
            if inter >= .05 * g.area: nfr += 1
            best = max(best, inter / g.union(pj).area)
        ious.append(best); frag.append(nfr)
    ious = np.array(ious)
    idmap = lambda d: rasterize([(g, i + 1) for i, g in enumerate(d.to_crs(crs).geometry.values)],
                                out_shape=(H, W), transform=transform, fill=0, dtype='int32')
    GT, P = idmap(g6), idmap(p6)
    def bnd(m):
        b = np.zeros(m.shape, bool); b[:, :-1] |= m[:, :-1] != m[:, 1:]; b[:-1] |= m[:-1] != m[1:]; return b
    gb, pb = bnd(GT) & (GT > 0), bnd(P) & (P > 0)
    dgt, dpb = ndi.distance_transform_edt(~gb), ndi.distance_transform_edt(~pb)
    dom = ndi.binary_dilation(GT > 0, iterations=3)
    prec = (dgt[pb & dom] <= 2).mean() if (pb & dom).any() else 0.0
    rec = (dpb[gb] <= 2).mean() if gb.any() else 0.0
    bF = 2 * prec * rec / (prec + rec + 1e-9)
    cov = ((P > 0) & (GT > 0)).sum() / max((GT > 0).sum(), 1)
    res = dict(label=label, gt=len(g6), pred=len(pred), IoU=round(float(ious.mean()), 3) if len(ious) else 0,
               IoU07=round(float(np.mean(ious >= .7)), 2) if len(ious) else 0, frag=round(float(np.mean(frag)), 2),
               merged=int((merged >= 2).sum()), bF=round(float(bF), 3), P=round(float(prec), 2), R=round(float(rec), 2),
               cover=round(float(cov), 2))
    if type_col in gt.columns:
        gtt = gt.to_crs(crs)
        for t in sorted(gtt[type_col].dropna().unique()):
            m = rasterize([(g, 1) for g in gtt[gtt[type_col] == t].geometry.values], out_shape=(H, W),
                          transform=transform, fill=0, dtype='uint8') > 0
            res[f'cover_{t}'] = round(float((P[m] > 0).mean()), 2) if m.any() else None
    print(' | '.join(f'{k} {v}' for k, v in res.items()))
    return res


def show_overlay(img_path, layers, title='', bbox=None, out_png=None, bands=(1, 2, 3), figsize=(11, 11)):
    """Картинка: подложка — ваш растр (3 канала, растяжка 2–98 %), поверх контуры слоёв.
    layers — список (GeoDataFrame, цвет, подпись)."""
    import matplotlib.pyplot as plt
    from rasterio.windows import from_bounds
    with rasterio.open(img_path) as s:
        win = from_bounds(*bbox, s.transform) if bbox is not None else rasterio.windows.Window(0, 0, s.width, s.height)
        k = max(1.0, max(win.width, win.height) / 2500)                  # большие охваты читаются с прореживанием
        shp = (len(bands), int(win.height / k), int(win.width / k))
        a = s.read(list(bands), window=win, boundless=True, out_shape=shp).astype(float)
        t = s.window_transform(win) * rasterio.Affine.scale(win.width / shp[2], win.height / shp[1]); crs = s.crs
    for i in range(a.shape[0]):
        lo, hi = np.nanpercentile(a[i][a[i] > 0], [2, 98]) if (a[i] > 0).any() else (0, 1)
        a[i] = np.clip((a[i] - lo) / (hi - lo + 1e-9), 0, 1)
    ext = (t.c, t.c + t.a * a.shape[2], t.f + t.e * a.shape[1], t.f)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(np.moveaxis(a, 0, -1), extent=ext)
    for gdf, color, name in layers:
        if gdf is not None and len(gdf):
            gdf.to_crs(crs).boundary.plot(ax=ax, color=color, linewidth=.8, label=name)
    ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3]); ax.set_title(title); ax.legend(loc='upper right')
    if out_png:
        fig.savefig(out_png, dpi=130, bbox_inches='tight')
    plt.show()

## Шаг 7 (по желанию). Сравнение с разметкой и картинка

In [ ]:
if GT_PATH:
    gt = load_gt(DRIVE_ROOT / GT_PATH, AOI_BBOX); kw = {}
    if COMPOSITE_PATH:
        with rasterio.open(DRIVE_ROOT / COMPOSITE_PATH) as c:
            kw = dict(transform=c.transform, shape=(c.height, c.width), crs=c.crs)
    rows = [evaluate(g[g.intersects(shapely.box(*gt.to_crs(4326).total_bounds))], gt, type_col=GT_TYPE_COL,
                     label=f'DA {w}', **kw) for w, g in RESULTS.items()]
    pd.DataFrame(rows).to_csv(OUT / f'{NAME}_DA_vs_gt.csv', index=False); display(pd.DataFrame(rows))
    if COMPOSITE_PATH:
        with rasterio.open(DRIVE_ROOT / COMPOSITE_PATH) as c:
            vb = transform_bounds('EPSG:4326', c.crs, *gt.to_crs(4326).total_bounds)
        w0 = list(RESULTS)[0]
        show_overlay(DRIVE_ROOT / COMPOSITE_PATH, [(RESULTS[w0], 'cyan', f'DA {w0}'), (gt, 'red', 'эталон')], title=NAME,
                     bbox=vb, out_png=OUT / f'{NAME}_DA_view.png')

## Что дальше
- Сравните окна дат между собой и с FTW (ноутбуки 01–02); лучшее окно запомните для всей области.
- Если модель хорошо видит границы, но режет/склеивает поля иначе, чем вам нужно, — подайте её границы в конвейер v7
  (`EXT_BOUNDARY['TIF']`): нарезку на поля по вашему определению сделает конвейер.
- Дообучение Delineate Anything возможно только общим обучением YOLO (Ultralytics) на разметке в формате YOLO-сегментации;
  для этого проекта проще и надёжнее дообучать FTW U-Net (ноутбук 04).